# Lecture 04 — Momento 2 : Word Embeddings (Word2Vec, GloVe, FastText)
Objetivo: **entender y usar** embeddings para similitud, analogías, visualización y un mini-prototipo.

## Actividades:
- Repaso  BoW vs Embeddings
- Entrenar Word2Vec
- Usar embeddings preentrenados (GloVe) + similitud/analogías
- FastText (subwords) y manejo de OOV + mini-proyecto


## 1) BoW vs Embeddings (repaso)
BoW/TF‑IDF: vector **disperso** de conteos/pesos por palabra.
Embeddings: vector **denso** (p.e. 50–300 dims) aprendido para que palabras con contextos parecidos queden cerca.


In [ ]:
!pip3 install gensim

In [ ]:
# representación clásica en BoW
from sklearn.feature_extraction.text import CountVectorizer
corpus = ["el gato está en la casa", "el perro está en el jardín"]
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)
print(X.toarray())
print(vectorizer.get_feature_names_out())


## 2) Entrenamiento Word2Vec
Entrenamos con pocas frases solo para ver el flujo.


In [ ]:
# # representaciónword2vec

from gensim.models import Word2Vec

sentences = [
    ["el", "gato", "está", "en", "la", "casa"],
    ["el", "perro", "está", "en", "el", "jardín"]
]
model = Word2Vec(sentences, vector_size=50, window=2, min_count=1, sg=1)

print(model.wv["gato"])
print(model.wv.similarity("gato", "perro"))


### Reto corto 1
1. Cambia `window` (2→5) y `vector_size` (50→100).
2. Observa cómo cambia `similarity("gato","perro")`.
3. Explica en 2–3 líneas **por qué** cambia.


## 3) Embeddings preentrenados (GloVe)
**No se entrena**: cargamos vectores ya aprendidos en un gran corpus.
Probar *most_similar*, analogías y una visualización 2D.


In [ ]:
# carga de modelos preentrenados
# GloVe

from gensim.downloader import load
model = load("glove-wiki-gigaword-50")  # vectores de 50 dimensiones
print(model["king"])
print(model.most_similar("king"))


In [ ]:
print(model["python"])
print(model.most_similar("python"))

In [ ]:
# Visualización 2D (PCA)
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

words = ["king", "queen", "man", "woman", "paris", "france"]
words = [w for w in words if w in model]  # por si alguna no está
vectors = [model[w] for w in words]

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(vectors)

for word, (x, y) in zip(words, coords):
    plt.scatter(x, y)
    plt.text(x+0.02, y+0.02, word)
plt.title("Proyección 2D (PCA) de embeddings")
plt.show()


In [ ]:
# Analogías: king - man + woman ≈ queen
model.most_similar(positive=["king","woman"], negative=["man"], topn=5)


### Reto corto 2
**Analogías:** prueba 3 analogías y comenta si funcionan.
- `paris - france + italy ≈ ?`
- `bigger - big + small ≈ ?`
- tu propia analogía

Adiciona una celda con resultados + 4–5 líneas de reflexión sobre **por qué** algunas analogías fallan.


In [ ]:
# ejercicio de Reto corto 2:
'''
Ejemplo1:
Dado un conjunto de palabras ‘similares’ en contexto, detectar las palabras fuera del contexto:
palabras = ["pera", "piña", "papaya", "carro"]

respuesta: "carro"

palabras = ["peach", "pineapple", "papaya", "car"]

respuesta: "car"

## cual es la sensibilidad al idioma?

'''

In [ ]:
# 'Odd one out' (palabra fuera de contexto)
# Idea: la palabra más 'lejana' al centroide del grupo suele ser la intrusa.
import numpy as np

def odd_one_out(words, kv):
    ws=[w for w in words if w in kv]
    if len(ws)<3:
        return None, ws
    X=np.stack([kv[w] for w in ws])
    c=X.mean(axis=0)
    # distancia coseno al centroide
    Xn=X/np.linalg.norm(X,axis=1,keepdims=True)
    cn=c/np.linalg.norm(c)
    cos = Xn@cn
    idx=np.argmin(cos)  # menor coseno = más lejos
    return ws[idx], ws

odd_one_out(["peach","pineapple","papaya","car"], model)


## 4) FastText (subwords) para OOV y morfología
FastText aprende embeddings **de n‑gramas de caracteres**.
Resultado: puede producir vector para palabras no vistas (OOV), y capta sufijos/prefijos.

En clase, para hacerlo rápido, entrenamos un **FastText mini** con pocas frases.


In [ ]:
from gensim.models import FastText

sentences = [
    "el gato negro duerme en la casa".split(),
    "el perro negro corre en el jardin".split(),
    "la gata blanca duerme".split(),
    "los perros corren rapido".split(),
    "me gusta programar en python".split(),
    "programacion y programar son similares".split(),
]

ft = FastText(
    sentences,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,
    min_n=3, max_n=5,  # subwords
    epochs=50,
)

ft.wv.similarity("programar","programacion")


In [ ]:
# OOV: palabra mal escrita o no vista
for w in ["programar", "programacion", "programación", "progrmar"]:
    try:
        v=ft.wv[w]
        print(w, "OK", "norm=", float(np.linalg.norm(v)))
    except KeyError:
        print(w, "OOV en vocab")

# Nota: en FastText, aunque una palabra no esté en el vocab,
# gensim puede generar vector si tiene subwords conocidos.


### Reto corto 3
1. Agrega 8–10 oraciones nuevas sobre un dominio (p.e. ciclismo, bases de datos, cine).
2. Entrena de nuevo `ft` (sube `epochs` si hace falta).
3. Demuestra un caso donde FastText ayude con variaciones morfológicas u OOV:
   - singular/plural, masculino/femenino, sufijos (`-ción`, `-mente`) o un typo.

Adiciona 1 celda con 3 similitudes (coseno) + 3–4 líneas de interpretación.


In [ ]:
# Mini-proyecto: buscador semántico de sinónimos locales
# Dado un término, devuelve las K palabras más cercanas según embeddings.

def semantic_neighbors(query, kv, topn=8):
    if query not in kv:
        return []
    return kv.most_similar(query, topn=topn)

semantic_neighbors("python", model, topn=10)


# Actividad Extraclase

## clustering con representación de embeddings y modelos clásicos no supervisados de ML

In [ ]:
# descargar y descomprimir modelo fasttext con corpus en español
# 1.3 GB
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.es.300.vec.gz

# 4.5 GB
!gunzip cc.es.300.vec.gz

In [ ]:
# cargar el modelo desde el archivo descargado previamente
import gensim

# Cargar modelo FastText para español (debe estar descargado)
model = gensim.models.KeyedVectors.load_word2vec_format("cc.es.300.vec", binary=False)


In [ ]:
palabras = [
    # objetos de casa
    "mueble", "nevera", "lavadora", "silla", "sofa", "tv",
    # colores
    "rojo", "amarillo", "verde", "azul", "gris", "negro",
    # Vehículos
    "auto", "camión", "moto", "bicicleta", "avión", "barco"
]

In [ ]:
# verificar que las palabras si estén en el corpus del modelo, filtrar las que no están
X = [model[word] for word in palabras if word in model]
palabras_filtradas = [word for word in palabras if word in model]
print(palabras_filtradas)

In [ ]:
# ejecutar un modelo clásico de kmeans con 3 grupos
from sklearn.cluster import KMeans

k = 3  # número esperado de grupos
kmeans = KMeans(n_clusters=k, random_state=42)
labels = kmeans.fit_predict(X)


In [ ]:
# visualizar en 2D grupos
# se utiliza PCA para poder proyectar en un espacio 2D vectores densos y conservar las relaciones de distancia
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Reducción de dimensión
pca = PCA(n_components=2)
coords = pca.fit_transform(X)

# Visualización
plt.figure(figsize=(10, 7))
for i, word in enumerate(palabras_filtradas):
    plt.scatter(coords[i, 0], coords[i, 1], c=f"C{labels[i]}")
    plt.text(coords[i, 0] + 0.01, coords[i, 1] + 0.01, word)
plt.title("Visualización de distancias WE+PCA+K-Means)")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.grid(True)
plt.show()

In [ ]:
# analice los resultados y plantee otro dataset de palabras que SI permita tener 3 grupos bien identificados
